# vLLM inference — pipeline integration test

Uses `OpenAICompatibleClient` (same as the real pipeline) with credentials
from `.env` pointing at the RunPod endpoint.

In [1]:
import sys, os
sys.path.insert(0, os.path.join("..", "src"))

from dotenv import load_dotenv
load_dotenv("../.env")

# OPENAI_MODEL is empty in .env — override with the --served-model-name from serve_vllm.sh
# os.environ.setdefault("OPENAI_MODEL", "Qwen/Qwen3-4B")

from app.api_client.openai_compatible_client import OpenAICompatibleClient
from app.api_client.base import Message

client = OpenAICompatibleClient()
print(f"base_url : {client._client.base_url}")
print(f"model    : {client.model}")

base_url : https://mz2x7t4w8upc14-8017.proxy.runpod.net/v1/
model    : Qwen/Qwen3-4B


In [2]:
# --- 1. Confirm the model is reachable -----------------------------------
models = client._client.models.list()
print("Served models:", [m.id for m in models.data])

Served models: ['Qwen/Qwen3-4B']


In [2]:
# --- 2. Plain text call + reasoning extraction ---------------------------
result = client.call([
    Message(role="system", content="You are a helpful assistant."),
    Message(role="user",   content="What is 7 * 8? Just give the number. /think"),
])

print("Answer   :", result.content)
print("Reasoning:", result.reasoning[:300] if result.reasoning else "(none)")

ChatCompletion(id='chatcmpl-9e9b30a0bfe7c8c1', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\n56', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning="\nOkay, the user is asking what 7 multiplied by 8 is, and they just want the number. Let me think. I know that 7 times 8 is a basic multiplication fact. Let me verify it. 7 times 8... Hmm, 7 times 10 is 70, so subtract 7 times 2, which is 14. 70 minus 14 is 56. Yeah, that makes sense. Alternatively, I can count 7 eight times: 7, 14, 21, 28, 35, 42, 49, 56. That's seven times, so the answer is 56. I should just give the number without any explanation since the user asked for it directly. So the answer is 56.\n"), stop_reason=None, token_ids=None)], created=1781973337, model='Qwen/Qwen3-4B', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=182, prompt_tokens=34, total

In [3]:
# --- 3. Structured output — same schema the generator uses ---------------
from app.generator.prompts import (
    GENERATOR_SYSTEM_PROMPT,
    GENERATE_USER_TEMPLATE,
    HypothesesResponse,
)
from app.models import RetrieverResult

mock_retriever_output = RetrieverResult(
    content=(
        "Evidence summary:\n"
        "- Induction heads form during a sharp phase transition in transformer training (Olsson et al. 2022).\n"
        "- Apparent emergent abilities may be an artefact of discontinuous metrics (Schaeffer et al. 2023).\n"
        "- Model scale correlates with sudden capability gains across diverse tasks (Wei et al. 2022)."
    )
)

question = "Why do large language models exhibit emergent abilities at scale?"

result = client.call(
    [
        Message(role="system", content=GENERATOR_SYSTEM_PROMPT),
        Message(role="user",   content=GENERATE_USER_TEMPLATE.format(
            prompt=question,
            retriever_output=mock_retriever_output.content,
        )),
    ],
    response_schema=HypothesesResponse,
)

print("Reasoning:\n", result.reasoning[:500] if result.reasoning else "(none)", "\n")
print("Hypotheses:")
for i, h in enumerate(result.content.hypotheses, 1):
    print(f"  {i}. {h}")

Reasoning:
 
Okay, let's tackle this. The user is asking why large language models exhibit emergent abilities at scale. The evidence given includes things like induction heads forming during a phase transition, emergent abilities possibly being an artefact of discontinuous metrics, and model scale correlating with sudden capability gains.

First, I need to make sure each hypothesis is clear and directly related to the prompt. They need to be grounded in the evidence, so I can't just make up stuff. Let me th 

Hypotheses:
  1. Large language models exhibit emergent abilities at scale due to a phase transition in training that enables the formation of induction heads, which unlock new computational modes capable of processing complex, context-dependent tasks that were previously inaccessible to smaller models.
  2. The apparent sudden capability gains in large language models are an artefact of discontinuous metric evaluations, where non-linear improvements in model performance are misin